# Heliostream on real data — OMNI training + MagNet comparison

This notebook takes the Heliostream project off the synthetic simulator and runs
it on **real solar-wind and Dst measurements** from NASA's OMNI database, then
puts the result next to the public **NOAA/NASA MagNet** benchmark for the same
task (forecasting the Dst geomagnetic-storm index from upstream solar wind).

It reuses the exact model, physics, training, and evaluation code from the
`heliostream` package, so this run also validates that shipped code against real
data (the environment the package was built in could not reach NASA).

**What runs here**
1. Fetch ~12 years of hourly OMNI data (solar wind + Dst).
2. Train the physics-informed hybrid and a black-box GRU baseline.
3. Evaluate with storm-weighted RMSE and conformal-calibrated intervals, against
   persistence and the empirical O'Brien-McPherron physics model.
4. Compare honestly to the MagNet benchmark (15.2 nT) and winners (~11 nT).
5. Probe robustness to simulated sensor dropout.

CPU is enough here (the model is a small GRU; the dataset is ~100k hours). A GPU
runtime will not change the results, only the wall-clock a little.

## 1. Setup

Install the data client, then load the `heliostream` package. Upload the `heliostream.zip` from the chat when prompted (or clone your repo — see the commented line).

In [ ]:
!pip install -q hapiclient


In [ ]:
import glob
# Get the heliostream package. Default: upload the zip from the chat.
if not glob.glob('heliostream/heliostream/__init__.py'):
    try:
        from google.colab import files
        print('Upload heliostream.zip …')
        up = files.upload()
        name = list(up)[0]
        !unzip -oq "{name}"
    except Exception as e:
        print('Upload path unavailable, trying git clone:', e)
        # !git clone https://github.com/ralegionc/heliostream.git
!pip install -q -e heliostream

# On Colab the unzipped folder named 'heliostream' can shadow the real package
# nested inside it. Put the real package first on the path and clear any cached
# (empty) namespace import so `heliostream.data` resolves correctly.
import os, sys
proj = os.path.abspath('heliostream')
sys.path.insert(0, proj)
for m in [x for x in list(sys.modules) if x == 'heliostream' or x.startswith('heliostream.')]:
    del sys.modules[m]
import heliostream
assert glob.glob(os.path.join(proj, 'heliostream', 'data', '__init__.py')), 'package layout unexpected'
print('package ready →', heliostream.__file__)


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())


## 2. Fetch real OMNI data

`OMNI2_H0_MRG1HR` is NASA/GSFC's definitive hourly merged solar wind, time-shifted
to Earth's bow shock, with the Dst index (`DST1800`). We pull 2012–2024, which
spans solar-cycle 24's activity and several strong storms (e.g. the March 2015 St
Patrick's Day storm, Dst ≈ −223 nT).

The loader resolves column names from the server metadata and cleans fill values.

In [ ]:
from heliostream.data import omni

df = omni.fetch(start='2012-01-01', stop='2024-01-01')   # first call downloads; then cached
print('rows:', len(df), '| span:', df.index.min().date(), '→', df.index.max().date())
print('storm hours  Dst<-50 :', int((df.dst < -50).sum()))
print('intense      Dst<-100:', int((df.dst < -100).sum()))
print('min Dst on record here:', round(float(df.dst.min()), 1), 'nT')
df.head()


In [ ]:
# Dst timeline with storm levels
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.plot(df.index, df.dst, lw=0.4, color='#7c5cff')
ax.axhline(-50, color='#c9a227', ls='--', lw=0.7, label='storm (-50)')
ax.axhline(-100, color='#ff8c42', ls='--', lw=0.7, label='intense (-100)')
ax.set_ylabel('Dst (nT)'); ax.set_title('Real OMNI Dst, 2012–2024')
ax.legend(loc='lower left', fontsize=8); plt.tight_layout(); plt.show()


## 3. Train on real data

Same code as the package. `train_model` splits chronologically (70/15/15), fits a
normalizer on the train span only (no leakage), and trains with a heteroscedastic
Gaussian likelihood so the model reports its own uncertainty. We up-weight storm
hours so the rare severe events are not drowned out by quiet time.

In [ ]:
from heliostream import train as T

hybrid, norm, hist_h = T.train_model(df, 'hybrid', epochs=60, storm_weight=1.0)
T.save(hybrid, norm, 'hybrid', hist_h, extra={'source': 'omni'})
print('hybrid trained. best val RMSE: %.2f nT' % min(h['val_rmse'] for h in hist_h))


In [ ]:
gru, norm_g, hist_g = T.train_model(df, 'gru', epochs=60, storm_weight=1.0)
T.save(gru, norm_g, 'gru', hist_g, extra={'source': 'omni'})
print('gru trained. best val RMSE: %.2f nT' % min(h['val_rmse'] for h in hist_g))


## 4. Evaluate + calibrate

Metrics on the held-out **test** span; conformal factors fit on **validation** so
the intervals carry a coverage guarantee to genuinely unseen data. We compare the
learned models to two references: persistence (carry the last Dst forward) and the
empirical O'Brien-McPherron physics model.

In [ ]:
from heliostream.data import features as F
from heliostream import evaluate as E

dfe = F.engineer(df)
_, va, te = F.time_split(dfe)

rep_h, (mh, sh, yh, factors) = E.full_report(hybrid, norm, va, te)
rep_g, _                     = E.full_report(gru, norm_g, va, te)
E.save_calibration(factors)

def row(name, r):
    return {'model': name,
            'RMSE (nT)': round(r['rmse_all'], 2),
            'storm RMSE (nT)': round(r['rmse_storm'], 2) if r['rmse_storm'] else np.nan,
            'MAE (nT)': round(r['mae_all'], 2)}

table = pd.DataFrame([
    row('hybrid (physics-informed)', rep_h['model']),
    row('GRU (black-box)',           rep_g['model']),
    row('O\u2019Brien\u2013McPherron (physics only)', rep_h['obm_empirical']),
    row('persistence',               rep_h['persistence']),
])
print('conformal coverage@90  hybrid=%.3f  gru=%.3f  (target 0.90)'
      % (rep_h['conformal']['coverage_conformal'], rep_g['conformal']['coverage_conformal']))
table


In [ ]:
# diagnostic plots (storm trace + reliability), rendered inline
from IPython.display import Image, display
p1, p2 = E.save_plots(hybrid, norm, te, factors, prefix='hybrid_omni')
display(Image(p1)); display(Image(p2))


In [ ]:
# per-horizon RMSE: where does skill decay with lead time?
h = rep_h['model']['rmse_per_h']; g = rep_g['model']['rmse_per_h']
x = np.array(range(1, len(h) + 1))
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(x, h, 'o-', color='#7c5cff', label='hybrid')
ax.plot(x, g, 's--', color='#8b88a8', label='GRU')
ax.set_xlabel('lead time (hours)'); ax.set_ylabel('RMSE (nT)')
ax.set_title('Forecast skill vs lead time'); ax.legend(); plt.tight_layout(); plt.show()


## 5. Where this sits vs the MagNet benchmark

NOAA/NASA's **MagNet** challenge is the public scoreboard for this exact task.
For reference, its LSTM benchmark scored **15.2 nT RMSE** and the top four
prize-winners reached **11.1–11.5 nT** on the held-out test set.

**Read this comparison honestly.** It is the same task and the same input
variables, but *not* the identical evaluation:
- MagNet scored a scaled Dst at t and t+1h from **minute-cadence** RTSW data on
  their specific frozen test set. This notebook uses **hourly** OMNI data, a
  multi-horizon (1–6 h) target, and a different chronological split.
- OMNI is *definitive* and time-shifted to the bow shock, i.e. cleaner than the
  real-time feed the operational MagNet setting cares about.

So treat the bar chart below as "are we in the right ballpark of published skill",
**not** as a leaderboard entry. To make a literal leaderboard claim you would run
the model inside MagNet's own data + scoring harness (a good follow-up).

In [ ]:
rmse_all = {
    'persistence':        rep_h['persistence']['rmse_all'],
    'O\u2019Brien\u2013McPherron': rep_h['obm_empirical']['rmse_all'],
    'GRU (black-box)':    rep_g['model']['rmse_all'],
    'hybrid (ours)':      rep_h['model']['rmse_all'],
}
fig, ax = plt.subplots(figsize=(8, 3.6))
bars = ax.bar(list(rmse_all), list(rmse_all.values()),
              color=['#3a3a52', '#5a5878', '#8b88a8', '#7c5cff'])
ax.axhspan(11.1, 11.5, color='#3ee6c4', alpha=0.18, label='MagNet winners (~11 nT)')
ax.axhline(15.2, color='#ff8c42', ls='--', lw=1, label='MagNet benchmark (15.2 nT)')
ax.set_ylabel('overall RMSE (nT)'); ax.set_title('Heliostream vs MagNet reference')
ax.legend(fontsize=8); plt.xticks(rotation=12, ha='right'); plt.tight_layout(); plt.show()


## 6. Robustness to sensor dropout

Operational feeds have gaps and the upstream spacecraft can change. A model that
only works on clean input is not useful. Here we corrupt a growing fraction of the
Bz channel in the test window (simulating a sensor gap held at its last value) and
measure how much each model's error grows. The physics prior should degrade more
gracefully than the black box.

In [ ]:
def dropout_rmse(model, normz, base_df, frac, seed=0):
    d = base_df.copy()
    rng = np.random.default_rng(seed)
    mask = rng.random(len(d)) < frac
    bz = d['bz'].values.copy()
    bz[mask] = np.nan
    d['bz'] = pd.Series(bz, index=d.index).ffill().bfill()   # naive gap-fill
    mean, std, y, _ = E.predict(model, d, normz)
    return float(np.sqrt(np.mean((mean - y) ** 2)))

fracs = [0.0, 0.1, 0.2, 0.3, 0.4]
te_raw = df.loc[te.index[0]:te.index[-1]]         # raw (un-engineered) test slice
hy = [dropout_rmse(hybrid, norm, te_raw, f) for f in fracs]
gr = [dropout_rmse(gru, norm_g, te_raw, f) for f in fracs]

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot([f*100 for f in fracs], hy, 'o-', color='#7c5cff', label='hybrid')
ax.plot([f*100 for f in fracs], gr, 's--', color='#8b88a8', label='GRU')
ax.set_xlabel('Bz samples dropped (%)'); ax.set_ylabel('test RMSE (nT)')
ax.set_title('Graceful degradation under sensor dropout'); ax.legend()
plt.tight_layout(); plt.show()
pd.DataFrame({'dropout %': [f*100 for f in fracs], 'hybrid RMSE': np.round(hy,2), 'GRU RMSE': np.round(gr,2)})


## 7. Save + honest summary

The trained real-data models are in `heliostream/artifacts/models/`. Point the
live dashboard at them with `python -m heliostream serve` on a networked machine.

**What this notebook establishes**
- The shipped code trains and evaluates on real OMNI data end to end.
- Storm-weighted RMSE, per-horizon skill, and conformal coverage are reported on a
  held-out span, with persistence and the empirical physics model as references.
- The result is placed in context against the MagNet benchmark (honestly, not as a
  literal leaderboard entry).

**What is still open (say this in the writeup)**
- A literal MagNet leaderboard number needs their exact minute-cadence data and
  scoring harness.
- Real *operational* skill needs evaluation on the noisier real-time NOAA feed,
  not just definitive OMNI.
- Beyond ~1 h lead, the forecast leans on solar-wind persistence; true multi-hour
  warning needs solar imagery (coronagraphs), which is out of scope here.

Download the trained model bundle if you want to keep it:

In [ ]:
import shutil
shutil.make_archive('heliostream_omni_models', 'zip', 'heliostream/artifacts/models')
try:
    from google.colab import files
    files.download('heliostream_omni_models.zip')
except Exception:
    print('saved heliostream_omni_models.zip')
